In [3]:
import docx

def extract_text_from_docx(docx_path):
    # Open the Word document
    doc = docx.Document(docx_path)
    # Extract text from every paragraph
    full_text = [para.text for para in doc.paragraphs if para.text.strip()]
    return " ".join(full_text)

def load_job_description(txt_path):
    with open(txt_path, 'r', encoding='utf-8') as file:
        return file.read().strip()

print("Extraction functions for Word documents ready!")

Extraction functions for Word documents ready!


In [4]:
# Paths
resume_path = "data/sample_resume.docx"
jd_path = "data/job_desc.txt"

# Extract text
resume_text = extract_text_from_docx(resume_path)
jd_text = load_job_description(jd_path)

# Preview
print("=== RESUME SAMPLE (FIRST 300 CHARS) ===")
print(resume_text[:300])
print("\n" + "="*40 + "\n")
print("=== JOB DESCRIPTION (FIRST 300 CHARS) ===")
print(jd_text[:300])

=== RESUME SAMPLE (FIRST 300 CHARS) ===
Name: Abiral Pandey Email: abiral.pandey88@gmail.com Phone: 940-242-3303 Current Location: Woonsocket, Rhode Island Visa Status: US Citizen SUMMARY: Dynamic individual with 6 years of software development experience in design, development, deployment, maintenance, production and support of web - bas


=== JOB DESCRIPTION (FIRST 300 CHARS) ===
We are looking for a Data Scientist or ML Engineer with strong Python experience. The ideal candidate has hands-on experience in NLP, machine learning algorithms, scikit-learn, pandas, spaCy, and Git. Responsibilities include extracting text data, building predictive models, and deploying NLP soluti


In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK data dependencies
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # 1. Convert to lowercase
    text = text.lower()
    
    # 2. Remove punctuation, special characters, and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 3. Tokenize into individual words
    words = text.split()
    
    # 4. Remove stop words and lemmatize (convert words to root form)
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(cleaned_words)

print("Preprocessing function ready!")

Preprocessing function ready!


In [6]:
# Apply cleaning
clean_resume = clean_text(resume_text)
clean_jd = clean_text(jd_text)

# Preview cleaned outputs
print("=== CLEANED RESUME (FIRST 200 CHARS) ===")
print(clean_resume[:200])
print("\n" + "="*40 + "\n")
print("=== CLEANED JOB DESCRIPTION (FIRST 200 CHARS) ===")
print(clean_jd[:200])

=== CLEANED RESUME (FIRST 200 CHARS) ===
name abiral pandey email abiral pandey gmail com phone current location woonsocket rhode island visa status u citizen summary dynamic individual year software development experience design development


=== CLEANED JOB DESCRIPTION (FIRST 200 CHARS) ===
looking data scientist ml engineer strong python experience ideal candidate hand experience nlp machine learning algorithm scikit learn panda spacy git responsibility include extracting text data buil


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Place cleaned texts into a list
corpus = [clean_resume, clean_jd]

# 2. Convert text into TF-IDF numerical vectors
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# 3. Calculate Cosine Similarity between resume (index 0) and JD (index 1)
similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
match_score = round(similarity[0][0] * 100, 2)

print(f" RESUME MATCHING SCORE: {match_score}%")

 RESUME MATCHING SCORE: 5.01%


In [8]:
# Extract all unique words learned by the vectorizer
feature_names = vectorizer.get_feature_names_out()

# Get non-zero terms for resume and JD
resume_vector = tfidf_matrix[0].toarray().flatten()
jd_vector = tfidf_matrix[1].toarray().flatten()

resume_keywords = set([feature_names[i] for i in range(len(feature_names)) if resume_vector[i] > 0])
jd_keywords = set([feature_names[i] for i in range(len(feature_names)) if jd_vector[i] > 0])

# Find matching and missing keywords
matched_skills = resume_keywords.intersection(jd_keywords)
missing_skills = jd_keywords - resume_keywords

print("=== MATCHED KEYWORDS ===")
print(list(matched_skills)[:15])

print("\n=== MISSING KEYWORDS FROM JOB DESCRIPTION ===")
print(list(missing_skills)[:15])

=== MATCHED KEYWORDS ===
['engineer', 'solution', 'deploying', 'include', 'building', 'model', 'strong', 'data', 'git', 'experience', 'responsibility']

=== MISSING KEYWORDS FROM JOB DESCRIPTION ===
['ideal', 'candidate', 'learning', 'scientist', 'hand', 'extracting', 'nlp', 'ml', 'predictive', 'scikit', 'looking', 'panda', 'learn', 'python', 'text']


In [ ]:
# Prototype prompt generator for resume tailoring
def generate_tailoring_prompt(resume_text, missing_keywords):
    missing_str = ", ".join(list(missing_keywords)[:10])
    
    prompt = f"""
    You are an expert technical career coach and resume writer. 
    
    I am applying for a position that requires the following key skills currently missing from my resume:
    [{missing_str}]
    
    Here is my current resume summary and bullet points:
    ---
    {resume_text[:1500]}
    ---
    
    Instructions:
    1. Identify existing bullet points where missing skills can naturally and truthfully be integrated.
    2. Rewrite 3 to 5 high-impact bullet points using the Action Verb + Context + Result formula.
    3. Do NOT make up fake work experience or fake company names. Reframe my real tasks using the target keywords.
    """
    return prompt

# Test prompt generation
tailor_prompt = generate_tailoring_prompt(resume_text, missing_skills)
print("=== GENERATED LLM PROMPT ===")
print(tailor_prompt[:500] + "...\n[Ready to send to Gemini/OpenAI API]")